# Testing the finetuned model

In [ ]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

In [ ]:
def build_messages(example: str, use_system_prompt: bool = True) -> list[dict]:
    messages = [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]
    if not use_system_prompt:
        messages = [messages[-1]]

    return messages

In [ ]:
from vllm import LLM, SamplingParams
import os

In [ ]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)
sampling_params = SamplingParams(**SAMPLING)

In [ ]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT") #modify
USER = os.getenv("SLURM_JOB_USER") #modify
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
input_model = "Qwen/Qwen3-4B-Instruct-2507"
model_output_name = f"{input_model}_finetuned"
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

In [ ]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

## Inference with system prompt

In [ ]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

In [ ]:
print(outputs[0].outputs[0].text)

## Inference without system prompt

In [ ]:
outputs = llm.chat(build_messages(prompt, use_system_prompt=False), sampling_params, use_tqdm=True)

In [ ]:
print(outputs[0].outputs[0].text)